# 🏛️ Data Warehouse Design — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> A data warehouse is a library, not a filing cabinet. A filing cabinet stores raw documents exactly as they arrived. A library organizes everything by subject, cross-references it, and makes retrieval fast. The Star Schema is the layout: one central Fact table (the checkout log — who borrowed what, when, for how much) surrounded by Dimension tables (the catalog cards — who, what book, which branch). Queries join outward from the center. The schema is denormalized intentionally: the goal is read speed, not write efficiency. SCDs (Slowly Changing Dimensions) answer: "what was the customer's address *when they placed this order*?" — historical accuracy through careful version tracking.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Data Warehouse Design? The Visual Model](#1) |
| 2 | [Creating / Setup — Schema Building Blocks](#2) |
| 3 | [The Core API — Fact vs Dimension, Join Patterns](#3) |
| 4 | [Decision Map — When to Use Which Pattern](#4) |
| 5 | [Pattern 1: Star Schema](#5) |
| 6 | [Pattern 2: Snowflake Schema](#6) |
| 7 | [Pattern 3: SCD Type 1, 2, 3 (Slowly Changing Dimensions)](#7) |
| 8 | [Pattern 4: Fact Table Design — Measures & Grain](#8) |
| 9 | [Pattern 5: Partitioning & Clustering Strategy](#9) |
| 10 | [The Data Warehouse Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Data Warehouse Design? The Visual Model

```
STAR SCHEMA:

             dim_customer           dim_product
            ┌────────────┐         ┌────────────┐
            │ customer_sk│         │ product_sk │
            │ name       │         │ name       │
            │ region     │         │ category   │
            └─────┬──────┘         └──────┬─────┘
                  │                        │
                  └──────────┬─────────────┘
                        ┌────▼────────────┐
            dim_date ───► fact_orders      ◄─── dim_store
                        │ order_sk (PK)   │
                        │ customer_sk (FK)│
                        │ product_sk  (FK)│
                        │ date_sk     (FK)│
                        │ store_sk    (FK)│
                        │ --- MEASURES ---│
                        │ revenue        │  ← additive
                        │ quantity       │  ← additive
                        │ discount_pct   │  ← non-additive
                        └────────────────┘

OLAP vs OLTP:
  OLTP:  normalized (3NF), optimized for writes, current state only
         e.g., your production Postgres — every INSERT/UPDATE fast
  OLAP:  denormalized, optimized for reads (GROUP BY, aggregations)
         e.g., Redshift, Snowflake, BigQuery — queries over billions of rows

SCD TYPE 2 EXAMPLE:
  dim_customer: history of a customer who moved from NY to CA
  customer_sk  customer_id  city  start_date    end_date     is_current
  1001         C123         NY    2020-01-01   2023-06-14    0
  1002         C123         CA    2023-06-15   9999-12-31    1

  → Orders before 2023-06-15 join to sk=1001 (NY)
  → Orders after join to sk=1002 (CA)
  → "What region was the customer in when they ordered?" — answered correctly.
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — Schema Building Blocks

In [ ]:
# DATA WAREHOUSE SCHEMA BUILDING BLOCKS
# Using Python dataclasses to model the schema structure

from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any
from datetime import date

@dataclass
class DimensionRecord:
    """A row in a dimension table — the 'who/what/when/where' context."""
    surrogate_key: int         # warehouse-generated PK — stable across source changes
    natural_key: str           # business identifier from source (e.g., customer_id)
    attributes: Dict[str, Any] # the descriptive columns
    # SCD Type 2 fields
    effective_start: str = "2000-01-01"
    effective_end: str = "9999-12-31"  # open-ended = current record
    is_current: bool = True

@dataclass
class FactRecord:
    """A row in a fact table — the measurable business event."""
    fact_sk: int               # surrogate key for the fact itself
    foreign_keys: Dict[str, int]  # dimension surrogate keys: {"customer_sk": 1001, ...}
    measures: Dict[str, float]    # the numbers: revenue, quantity, etc.

@dataclass
class WarehouseTable:
    name: str
    table_type: str            # "fact" or "dimension"
    grain: str                 # one row = one what? e.g., "one order line item"
    rows: List = field(default_factory=list)

    def row_count(self):
        return len(self.rows)

    def describe(self):
        print(f"Table: {self.name} ({self.table_type})")
        print(f"  Grain: {self.grain}")
        print(f"  Row count: {self.row_count()}")
        if self.rows:
            first = self.rows[0]
            if hasattr(first, 'attributes'):
                print(f"  Attributes: {list(first.attributes.keys())}")
            if hasattr(first, 'measures'):
                print(f"  Measures: {list(first.measures.keys())}")

# Create a sample star schema
dim_customer = WarehouseTable("dim_customer", "dimension", "one current state per customer")
fact_orders = WarehouseTable("fact_orders", "fact", "one row per order line item")

dim_customer.rows.append(DimensionRecord(
    surrogate_key=1001, natural_key="C123",
    attributes={"name": "Alice", "city": "New York", "tier": "Gold"}
))
fact_orders.rows.append(FactRecord(
    fact_sk=1, foreign_keys={"customer_sk": 1001, "product_sk": 501, "date_sk": 20260320},
    measures={"revenue": 299.99, "quantity": 2, "discount_pct": 0.10}
))

print("=== Sample Star Schema ===")
dim_customer.describe()
print()
fact_orders.describe()
print("\nSchema building blocks defined.")

<a id='3'></a>
## 3. ⚡ The Core API — Fact vs Dimension, Join Patterns

```
CONCEPT            FACT TABLE                   DIMENSION TABLE
─────────────────────────────────────────────────────────────────────────
Contains           measurements (events)        context (attributes)
Rows represent     one business event           one entity or version
PK type            surrogate key (auto int)     surrogate key + natural key
Size               very large (billions)        small to medium (millions)
Changes            immutable (append-only)      changes via SCD
Indexes            foreign keys                 natural key, SCD dates
─────────────────────────────────────────────────────────────────────────

SURROGATE KEY vs NATURAL KEY:
  Natural key: the business identifier (customer_id = 'C123')
    Problem: source system can reuse, change format, or merge keys
  Surrogate key: warehouse-assigned integer (customer_sk = 1001)
    Benefits: stable reference even if source changes; SCD versioning

MEASURE TYPES:
  Additive:       revenue, quantity — can sum across ANY dimension
  Semi-additive:  balance, inventory — sum across some dims (not time)
  Non-additive:   ratios, percentages — NEVER sum directly (avg or ratio)

THINGS YOU DO NOT DO:
❌  Store non-additive measures as raw columns → analysts will sum them wrong
✅  Store numerator + denominator; compute ratio at query time
❌  Use natural key as fact table FK → breaks when source system changes
✅  Always join via surrogate key; natural key lives only in dimension table
❌  Put descriptive text in fact tables (customer_name in fact_orders)
✅  All context in dimension tables; fact has only FKs and measures
```

In [ ]:
# LIVE DEMO: star schema query pattern (Python simulation of SQL join)

# Simplified in-memory star schema
dim_customer_data = {
    1001: {"name": "Alice",   "region": "West",  "tier": "Gold"},
    1002: {"name": "Bob",     "region": "East",  "tier": "Silver"},
    1003: {"name": "Charlie", "region": "West",  "tier": "Bronze"},
}

dim_product_data = {
    501: {"name": "Laptop",  "category": "Electronics", "unit_cost": 800},
    502: {"name": "Desk",    "category": "Furniture",    "unit_cost": 200},
    503: {"name": "Monitor", "category": "Electronics", "unit_cost": 350},
}

fact_rows = [
    {"customer_sk": 1001, "product_sk": 501, "revenue": 999.99, "qty": 1},
    {"customer_sk": 1002, "product_sk": 502, "revenue": 399.99, "qty": 2},
    {"customer_sk": 1001, "product_sk": 503, "revenue": 699.99, "qty": 1},
    {"customer_sk": 1003, "product_sk": 501, "revenue": 999.99, "qty": 1},
    {"customer_sk": 1002, "product_sk": 503, "revenue": 349.99, "qty": 1},
]

# Simulate: SELECT region, SUM(revenue) FROM fact_orders JOIN dim_customer GROUP BY region
def query_revenue_by_region():
    region_revenue = {}
    for row in fact_rows:
        customer = dim_customer_data[row["customer_sk"]]   # join to dimension
        region = customer["region"]
        region_revenue[region] = region_revenue.get(region, 0) + row["revenue"]
    return region_revenue

# Simulate: SELECT category, SUM(revenue), AVG(unit_cost) GROUP BY category
def query_category_stats():
    stats = {}
    for row in fact_rows:
        product = dim_product_data[row["product_sk"]]
        cat = product["category"]
        if cat not in stats:
            stats[cat] = {"revenue": 0, "count": 0}
        stats[cat]["revenue"] += row["revenue"]
        stats[cat]["count"] += 1
    return {cat: {"total_rev": round(v["revenue"], 2), "orders": v["count"]}
            for cat, v in stats.items()}

print("Revenue by region:")
for region, rev in sorted(query_revenue_by_region().items()):
    print(f"  {region}: ${rev:,.2f}")

print("\nCategory stats:")
for cat, stats in query_category_stats().items():
    print(f"  {cat}: {stats}")

print("\nNon-additive measure trap:")
for row in fact_rows:
    product = dim_product_data[row["product_sk"]]
    # CORRECT: compute margin at row level, then aggregate
    margin = (row["revenue"] - product["unit_cost"] * row["qty"]) / row["revenue"]
    print(f"  {product['name']:10s} margin={margin:.1%}")  # do NOT SUM these percentages

<a id='4'></a>
## 4. 🗂️ Decision Map — When to Use Which Pattern

```
SCENARIO                              PATTERN CHOICE
──────────────────────────────────────────────────────────────────────
Business users query directly (BI)    Star Schema (simple joins, fast)
Many analysts, normalize storage      Snowflake Schema (3NF dimensions)
Attributes change but history needed  SCD Type 2 (new row per change)
Only current value matters            SCD Type 1 (overwrite in place)
Track both old + current + one prior  SCD Type 3 (add prev_value column)
Large fact table, date-based queries  Partition by date, cluster by FK
──────────────────────────────────────────────────────────────────────

STAR vs SNOWFLAKE:
  Star:       dim tables are denormalized (redundant columns OK)
              → fewer joins, faster queries, easier for BI tools
  Snowflake:  dim tables normalized (sub-dimensions separate)
              → more joins, saves storage, harder to query
  Recommendation: start with Star; snowflake only if storage is critical

SCD TYPE SELECTOR:
  History needed for fact accuracy?  → SCD Type 2 (most common)
  Just overwrite, no history?        → SCD Type 1
  Exactly one prior value needed?    → SCD Type 3
  Complex versioning?                → SCD Type 6 (Type 1+2+3 combined)

GRAIN RULE:
  Declare grain FIRST before adding columns.
  All measures must be true AT the grain level.
  All FKs must be fully determined by the grain.
  If unsure: go to lowest possible grain (can always roll up, never drill down).
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Star Schema

---

```
SCENARIO:  Design the DW schema for a retail company's order analytics.
           Business users need: revenue by region/product/date/store.

STAR SCHEMA DESIGN STEPS:
  1. Identify the business process: Order transactions
  2. Declare the grain: one row per ORDER LINE ITEM
     (not per order — one order can have multiple products)
  3. Identify dimensions: Customer, Product, Date, Store
  4. Identify measures: revenue, quantity, discount_amount
  5. Assign surrogate keys to all dimension tables
  6. Fact table = FKs to each dimension + measures

RESULTING SCHEMA:

  dim_date:     date_sk, date, year, quarter, month, day_of_week, is_holiday
  dim_customer: customer_sk, customer_id, name, region, city, tier
  dim_product:  product_sk, product_id, name, category, subcategory, brand
  dim_store:    store_sk, store_id, name, city, region, size_sqft

  fact_order_line:
    order_line_sk  (PK)
    order_id       (degenerate dimension — no separate table needed)
    date_sk        (FK → dim_date)
    customer_sk    (FK → dim_customer)
    product_sk     (FK → dim_product)
    store_sk       (FK → dim_store)
    revenue        (additive measure)
    quantity       (additive measure)
    discount_amount (additive measure — NOT discount_pct which is non-additive)

QUERY EXAMPLE:
  SELECT c.region, SUM(f.revenue)
  FROM fact_order_line f
  JOIN dim_customer c ON f.customer_sk = c.customer_sk
  JOIN dim_date d     ON f.date_sk = d.date_sk
  WHERE d.year = 2026
  GROUP BY c.region
  → Two joins only (vs. 5+ joins in normalized OLTP schema)
```

In [ ]:
# STAR SCHEMA BUILDER SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict, Set

@dataclass
class StarSchemaDesign:
    business_process: str
    grain: str
    dimensions: Dict[str, List[str]] = field(default_factory=dict)
    measures: List[str] = field(default_factory=list)
    degenerate_dims: List[str] = field(default_factory=list)  # IDs in fact with no dim table

    def add_dimension(self, name: str, columns: List[str]):
        # always add surrogate_key and natural_key first
        self.dimensions[name] = [f"{name}_sk", f"{name.split('_')[1] if '_' in name else name}_id"] + columns

    def generate_fact_table(self) -> Dict:
        fks = [f"{dim}_sk" for dim in self.dimensions]
        return {
            "name": f"fact_{self.business_process.lower().replace(' ', '_')}",
            "grain": self.grain,
            "columns": (
                ["fact_sk"] +
                fks +
                self.degenerate_dims +
                self.measures
            )
        }

    def describe(self):
        print(f"Business process: {self.business_process}")
        print(f"Grain: {self.grain}")
        print()
        print("DIMENSION TABLES:")
        for dim_name, cols in self.dimensions.items():
            print(f"  {dim_name}: {cols}")
        print()
        fact = self.generate_fact_table()
        print(f"FACT TABLE: {fact['name']}")
        print(f"  Grain: {fact['grain']}")
        print(f"  Columns: {fact['columns']}")

# Design the retail orders star schema
orders_schema = StarSchemaDesign(
    business_process="order_line_items",
    grain="one row per order line item (order + product combination)"
)

orders_schema.add_dimension("dim_date",     ["date", "year", "quarter", "month", "day_of_week", "is_holiday"])
orders_schema.add_dimension("dim_customer", ["name", "region", "city", "tier"])
orders_schema.add_dimension("dim_product",  ["name", "category", "subcategory", "brand", "unit_cost"])
orders_schema.add_dimension("dim_store",    ["name", "city", "region", "size_sqft"])

orders_schema.degenerate_dims = ["order_id"]    # in fact table, no separate dim needed
orders_schema.measures = ["revenue", "quantity", "discount_amount", "cost"]

orders_schema.describe()

print()
print("MEASURE CLASSIFICATION:")
measures = {"revenue": "additive", "quantity": "additive",
            "discount_amount": "additive", "discount_pct": "NON-ADDITIVE → store as amount instead"}
for m, classification in measures.items():
    print(f"  {m:20s} → {classification}")

<a id='6'></a>
## 6. 🧩 Pattern 2: Snowflake Schema

---

```
SCENARIO:  Product dimension has many attributes that create redundancy.
           "Electronics" appears millions of times in dim_product.
           Normalize categories into a separate dim_category table.

STAR vs SNOWFLAKE:

STAR (denormalized):
  dim_product: product_sk, product_id, name, category_name, category_dept, brand
  → "Electronics" stored once per product row (millions of times)

SNOWFLAKE (normalized):
  dim_product:  product_sk, product_id, name, category_sk (FK), brand
  dim_category: category_sk, category_id, category_name, department_sk (FK)
  dim_dept:     dept_sk, dept_id, dept_name
  → "Electronics" stored once in dim_category

QUERY COMPARISON:
  Star:  FROM fact JOIN dim_product ON ...            (1 join to get category)
  Snow:  FROM fact JOIN dim_product ON ...
              JOIN dim_category ON ...                (2 joins to get category)
              JOIN dim_dept ON ...                    (3 joins to get department)

TRADEOFFS:
  Snowflake pros:
    - Smaller storage footprint (no repeated strings in dim)
    - Easier to maintain category hierarchy independently
    - Consistent updates: change "Electronics" once, propagates everywhere

  Snowflake cons:
    - More joins → harder queries for business users
    - BI tools have to traverse hierarchy
    - Modern columnar storage (Redshift, Snowflake) often negates storage savings

RECOMMENDATION: Use Star Schema. Snowflake only if you have deep hierarchies
                with many levels (geography: city→state→country→continent).
```

In [ ]:
# SNOWFLAKE vs STAR: storage and query complexity comparison

# Star: dim_product stores all category info inline (denormalized)
star_dim_product = [
    {"product_sk": 501, "name": "Laptop 15",  "category": "Electronics", "dept": "Technology"},
    {"product_sk": 502, "name": "Laptop 13",  "category": "Electronics", "dept": "Technology"},
    {"product_sk": 503, "name": "USB Mouse",  "category": "Electronics", "dept": "Technology"},
    {"product_sk": 504, "name": "Standing Desk", "category": "Furniture", "dept": "Office"},
    {"product_sk": 505, "name": "Desk Chair", "category": "Furniture", "dept": "Office"},
]

# Snowflake: dim_product + dim_category + dim_dept (normalized)
snow_dim_dept = {1: "Technology", 2: "Office"}
snow_dim_category = {
    10: {"name": "Electronics", "dept_sk": 1},
    20: {"name": "Furniture",   "dept_sk": 2},
}
snow_dim_product = [
    {"product_sk": 501, "name": "Laptop 15",     "category_sk": 10},
    {"product_sk": 502, "name": "Laptop 13",     "category_sk": 10},
    {"product_sk": 503, "name": "USB Mouse",     "category_sk": 10},
    {"product_sk": 504, "name": "Standing Desk", "category_sk": 20},
    {"product_sk": 505, "name": "Desk Chair",    "category_sk": 20},
]

def star_query_by_dept(fact_rows):
    # Star: one join to get department
    prod_lookup = {p["product_sk"]: p for p in star_dim_product}
    dept_revenue = {}
    for row in fact_rows:
        dept = prod_lookup[row["product_sk"]]["dept"]   # 1 join
        dept_revenue[dept] = dept_revenue.get(dept, 0) + row["revenue"]
    return dept_revenue

def snowflake_query_by_dept(fact_rows):
    # Snowflake: THREE joins to get department
    prod_lookup = {p["product_sk"]: p for p in snow_dim_product}
    cat_lookup = snow_dim_category
    dept_lookup = snow_dim_dept
    dept_revenue = {}
    for row in fact_rows:
        product = prod_lookup[row["product_sk"]]         # join 1
        category = cat_lookup[product["category_sk"]]   # join 2
        dept = dept_lookup[category["dept_sk"]]         # join 3
        dept_revenue[dept] = dept_revenue.get(dept, 0) + row["revenue"]
    return dept_revenue

sample_facts = [{"product_sk": 501, "revenue": 999}, {"product_sk": 504, "revenue": 399},
                {"product_sk": 503, "revenue": 49},  {"product_sk": 502, "revenue": 799}]

print("Star query (1 join):",        star_query_by_dept(sample_facts))
print("Snowflake query (3 joins):",  snowflake_query_by_dept(sample_facts))
print("(Results identical — star is simpler to write and execute)")

# Storage: star repeats 'Electronics' and 'Technology' in every product row
star_redundant_bytes = sum(len(p["category"]) + len(p["dept"]) for p in star_dim_product)
snow_stored_once = sum(len(c["name"]) for c in snow_dim_category.values())
print(f"\nStar repeated string storage: {star_redundant_bytes} chars")
print(f"Snowflake stored-once storage: {snow_stored_once} chars")
print("In real DW (millions of product rows), snowflake saves significant storage")
print("But modern columnar compression often negates this advantage")

<a id='7'></a>
## 7. 🧩 Pattern 3: SCD Types — Slowly Changing Dimensions

---

```
PROBLEM:  A customer moves from New York to California. Their orders should
          reflect the region THEY WERE IN when they placed each order.

SCD TYPE 1 — OVERWRITE (no history):
  Update the row in place. Customer now shows California everywhere.
  Old orders appear to be from California (historical accuracy lost).
  Use when: history doesn't matter (e.g., fixing a typo in a name).
  customer_sk  city   is_current
  1001         CA     1          ← overwritten, NY gone forever

SCD TYPE 2 — NEW ROW (full history):
  Insert a new row for the change, close the old row.
  Old orders join to old SK (NY), new orders join to new SK (CA).
  Use when: historical accuracy is required (most common choice).
  customer_sk  nat_key  city  eff_start    eff_end      is_current
  1001         C123     NY    2020-01-01   2023-06-14   0  ← closed
  1002         C123     CA    2023-06-15   9999-12-31   1  ← current

  Fact rows from before 2023-06-15 point to sk=1001 (NY).
  Fact rows from after point to sk=1002 (CA). ✓

SCD TYPE 3 — ADD COLUMN (one prior value):
  Add prev_city column, update both columns on change.
  Can only track ONE previous value — can't go further back.
  customer_sk  city  prev_city
  1001         CA    NY

SCD TYPE 2 LOADING PATTERN:
  1. Hash all tracked attributes into a row_hash
  2. Compare incoming record hash to current dimension row hash
  3. If changed: CLOSE old row (set eff_end = today - 1 day, is_current = 0)
                 INSERT new row (eff_start = today, eff_end = 9999, is_current = 1)
  4. If unchanged: no action
```

In [ ]:
# SCD TYPE 2 IMPLEMENTATION SIMULATION

from dataclasses import dataclass, field
from typing import List, Dict, Optional
import hashlib
import json

@dataclass
class SCDRow:
    surrogate_key: int
    natural_key: str
    attributes: Dict
    effective_start: str
    effective_end: str
    is_current: bool
    row_hash: str = ""

    def __post_init__(self):
        if not self.row_hash:
            # hash of all tracked attributes — used to detect changes
            self.row_hash = hashlib.md5(
                json.dumps(self.attributes, sort_keys=True).encode()
            ).hexdigest()[:8]

class SCD2DimensionTable:
    def __init__(self, name: str):
        self.name = name
        self.rows: List[SCDRow] = []
        self._next_sk = 1000

    def _next_surrogate(self):
        self._next_sk += 1
        return self._next_sk

    def upsert(self, natural_key: str, new_attrs: Dict, effective_date: str):
        # find current row for this natural key
        current = next((r for r in self.rows
                        if r.natural_key == natural_key and r.is_current), None)

        new_hash = hashlib.md5(
            json.dumps(new_attrs, sort_keys=True).encode()
        ).hexdigest()[:8]

        if current is None:
            # first load — insert new row
            self.rows.append(SCDRow(
                surrogate_key=self._next_surrogate(),
                natural_key=natural_key,
                attributes=new_attrs,
                effective_start=effective_date,
                effective_end="9999-12-31",
                is_current=True
            ))
            print(f"  [INSERT] {natural_key} first load → sk={self._next_sk}")

        elif current.row_hash != new_hash:
            # attributes changed → close old row, insert new row
            current.effective_end = effective_date
            current.is_current = False
            old_sk = current.surrogate_key

            self.rows.append(SCDRow(
                surrogate_key=self._next_surrogate(),
                natural_key=natural_key,
                attributes=new_attrs,
                effective_start=effective_date,
                effective_end="9999-12-31",
                is_current=True
            ))
            print(f"  [SCD2 CHANGE] {natural_key}: close sk={old_sk}, new sk={self._next_sk}")
            print(f"    changed: {current.attributes} → {new_attrs}")
        else:
            print(f"  [NO CHANGE] {natural_key}")

    def current_row(self, natural_key: str) -> Optional[SCDRow]:
        return next((r for r in self.rows if r.natural_key == natural_key and r.is_current), None)

    def lookup_at_date(self, natural_key: str, event_date: str) -> Optional[SCDRow]:
        # find which version was current at the time of an event
        for r in self.rows:
            if (r.natural_key == natural_key and
                r.effective_start <= event_date <= r.effective_end):
                return r
        return None

# Demo: customer moves from NY to CA
dim_cust = SCD2DimensionTable("dim_customer")

print("Initial load (2020-01-01):")
dim_cust.upsert("C123", {"name": "Alice", "city": "New York", "tier": "Silver"}, "2020-01-01")

print("\nNo change (2021-01-01):")
dim_cust.upsert("C123", {"name": "Alice", "city": "New York", "tier": "Silver"}, "2021-01-01")

print("\nCustomer moves to CA, promoted to Gold (2023-06-15):")
dim_cust.upsert("C123", {"name": "Alice", "city": "California", "tier": "Gold"}, "2023-06-15")

print("\nAll rows in dim_customer:")
for row in dim_cust.rows:
    print(f"  sk={row.surrogate_key} city={row.attributes['city']:12s} "
          f"tier={row.attributes['tier']:6s} {row.effective_start}→{row.effective_end} current={row.is_current}")

# Test: what region was customer in for an order placed on 2022-03-01?
old_row = dim_cust.lookup_at_date("C123", "2022-03-01")
new_row = dim_cust.lookup_at_date("C123", "2024-01-01")
print(f"\nOrder on 2022-03-01 → customer was in: {old_row.attributes['city']}")
print(f"Order on 2024-01-01 → customer was in: {new_row.attributes['city']}")

<a id='8'></a>
## 8. 🧩 Pattern 4: Fact Table Design — Measures & Grain

---

```
SCENARIO:  Design fact table for a bank's transactions.
           Common mistakes: wrong grain, non-additive measures, missing FKs.

GRAIN OPTIONS for bank transactions:
  Option A: one row per TRANSACTION         (most granular)
  Option B: one row per ACCOUNT per DAY     (daily balance snapshot)
  Option C: one row per CUSTOMER per MONTH  (monthly summary)

  Choose A if: analysts need to drill to individual transactions
  Choose B if: "what was the balance on a given day" is the core query
  Choose C if: storage is critical and daily granularity not needed

MEASURES for transaction grain:
  amount:        ADDITIVE — sum across any dimension ✓
  running_balance: SEMI-ADDITIVE — sum across accounts but NOT time
                   (balance at Dec 31 ≠ sum of daily balances)
  interest_rate: NON-ADDITIVE — store as numerator/denominator or as-is,
                 never SUM (meaningless), use AVG weighted by amount

DEGENERATE DIMENSIONS:
  transaction_id: appears in fact table but no separate dim table
  order_id, invoice_number, check_number → all degenerate dimensions
  These are NOT measures (no arithmetic), but NOT rich enough for a dim table

FACTLESS FACT TABLES:
  Record events that have no numeric measures:
  "Student attended class" — no amount, just the event
  Fact row: student_sk, class_sk, date_sk, attendance_flag=1
  Count the rows = count of attendances
```

In [ ]:
# FACT TABLE DESIGN DEMO: measure type handling

# Simulated fact_transactions (grain = one row per transaction)
fact_transactions = [
    {"txn_id": 1, "account_sk": 101, "date_sk": 20260101, "amount": 500.0, "balance": 1500.0, "type": "deposit"},
    {"txn_id": 2, "account_sk": 101, "date_sk": 20260102, "amount": -200.0, "balance": 1300.0, "type": "withdrawal"},
    {"txn_id": 3, "account_sk": 102, "date_sk": 20260101, "amount": 1000.0, "balance": 5000.0, "type": "deposit"},
    {"txn_id": 4, "account_sk": 102, "date_sk": 20260103, "amount": -500.0, "balance": 4500.0, "type": "withdrawal"},
]

# ADDITIVE measure: sum(amount) across any dimension
total_deposits = sum(r["amount"] for r in fact_transactions if r["amount"] > 0)
total_withdrawals = sum(r["amount"] for r in fact_transactions if r["amount"] < 0)
print(f"Total deposits: ${total_deposits:,.2f}  (additive — correct to sum)")
print(f"Total withdrawals: ${total_withdrawals:,.2f}")

# SEMI-ADDITIVE measure: balance — DO NOT SUM across time
wrong_balance_sum = sum(r["balance"] for r in fact_transactions)
correct_balances = {}
for r in fact_transactions:
    # correct: take LAST balance per account (point-in-time)
    correct_balances[r["account_sk"]] = r["balance"]
correct_total = sum(correct_balances.values())

print(f"\nBalance handling:")
print(f"  WRONG SUM of all balance rows: ${wrong_balance_sum:,.2f}  ← never do this")
print(f"  CORRECT latest balance per account, then sum: ${correct_total:,.2f}")

# NON-ADDITIVE: interest_rate — store numerator + denominator separately
loan_facts = [
    {"loan_sk": 1, "principal": 10000, "interest_rate": 0.05},  # $500 interest
    {"loan_sk": 2, "principal": 50000, "interest_rate": 0.03},  # $1500 interest
]

# WRONG: average rate (unweighted)
wrong_avg_rate = sum(r["interest_rate"] for r in loan_facts) / len(loan_facts)

# CORRECT: weighted average by principal
total_interest = sum(r["principal"] * r["interest_rate"] for r in loan_facts)
total_principal = sum(r["principal"] for r in loan_facts)
correct_weighted_rate = total_interest / total_principal

print(f"\nInterest rate handling:")
print(f"  WRONG unweighted average: {wrong_avg_rate:.3%}")
print(f"  CORRECT weighted by principal: {correct_weighted_rate:.3%}")
print(f"  (DW best practice: store interest_amount and principal — compute rate at query time)")

<a id='9'></a>
## 9. 🧩 Pattern 5: Partitioning & Clustering Strategy

---

```
SCENARIO:  fact_orders has 5 billion rows. Queries always filter by date
           and group by region or product category. Optimize storage and query cost.

PARTITIONING (physical file organization):
  Partition by: order_date (most common for time-series facts)
  Result: s3://dw/fact_orders/order_date=2026-03-20/*.parquet
  Benefit: date-filtered queries read ONLY the relevant date partition
           → partition pruning cuts 99%+ of data scanned on typical queries

  Partition granularity:
    Too fine (hourly):  millions of tiny files → slow metadata, no compression
    Too coarse (yearly): one huge file → can't prune → scans too much
    Sweet spot (daily or monthly): depends on data volume and query patterns

CLUSTERING / SORT KEYS (within a partition):
  Cluster by: customer_sk, product_sk (most common filter after date)
  Benefit: co-located similar customer_sk values → block-level pruning
  Redshift SORTKEY: rows physically sorted on disk → zone maps skip blocks
  BigQuery CLUSTER BY: blocks sorted, each block's min/max stored in metadata

DISTRIBUTION KEYS (Redshift-specific — sharding across nodes):
  DISTKEY(customer_sk): all rows for same customer on same node → fast joins
  DISTSTYLE EVEN: round-robin distribution → balanced load, worse for joins
  DISTSTYLE ALL: broadcast small tables to all nodes → eliminates join shuffles

RULES OF THUMB:
  Partition by: time column (date, month) — used in WHERE clause of 90% of queries
  Cluster/sort by: columns used in JOIN or GROUP BY after the partition filter
  Distribution key: choose the column used in most frequent large table JOINs
  File size: aim for 128MB–1GB per partition file (sweet spot for most engines)
```

In [ ]:
# PARTITIONING STRATEGY DEMO

from dataclasses import dataclass, field
from typing import List, Dict
import math

@dataclass
class PartitionDesign:
    table_name: str
    total_rows: int
    partition_column: str
    partition_granularity: str   # "day", "month", "year"
    cluster_columns: List[str] = field(default_factory=list)
    row_size_bytes: int = 200

    def total_gb(self):
        return self.total_rows * self.row_size_bytes / 1e9

    def partition_count(self):
        if self.partition_granularity == "day":   return 365 * 3   # 3 years
        if self.partition_granularity == "month": return 12 * 3
        if self.partition_granularity == "year":  return 3
        return 1

    def partition_size_mb(self):
        return (self.total_gb() * 1000) / self.partition_count()

    def query_data_scanned_gb(self, date_filter_days: int):
        # with partition pruning: only scan the date range
        if self.partition_granularity == "day":
            return date_filter_days * self.partition_size_mb() / 1000
        return self.total_gb()   # no pruning possible

    def describe(self):
        print(f"Table: {self.table_name}")
        print(f"  Total: {self.total_rows/1e9:.1f}B rows, {self.total_gb():.1f} GB")
        print(f"  Partition: {self.partition_column} ({self.partition_granularity})")
        print(f"  Partition count: {self.partition_count():,}")
        print(f"  Avg partition size: {self.partition_size_mb():.0f} MB")
        print(f"  Cluster columns: {self.cluster_columns}")

# Compare partition strategies for a 5B row fact table
scenarios = [
    PartitionDesign("fact_orders_daily",   5_000_000_000, "order_date", "day",   ["customer_sk", "product_sk"]),
    PartitionDesign("fact_orders_monthly", 5_000_000_000, "order_date", "month", ["customer_sk"]),
    PartitionDesign("fact_orders_yearly",  5_000_000_000, "order_date", "year",  []),
]

for design in scenarios:
    design.describe()
    for days in [1, 7, 30]:
        scanned = design.query_data_scanned_gb(days)
        print(f"  Query last {days:2d} days → scans {scanned:.2f} GB")
    print()

print("RECOMMENDATION:")
print("  Daily partitioning: partition size ~55MB ← may be too small (many files)")
print("  Monthly partitioning: ~1.6GB per partition ← sweet spot")
print("  Yearly partitioning: ~19GB per partition ← too coarse, scans too much")
print("  Rule: aim for 128MB-1GB per partition file")

<a id='10'></a>
## 10. 🗺️ The Data Warehouse Decision Map

```
REQUIREMENT                              PATTERN                  NOTES
────────────────────────────────────────────────────────────────────────────
BI queries, simple joins                 Star Schema              default choice
Deep hierarchies, normalize storage      Snowflake Schema         rarely needed
Historical accuracy for changes          SCD Type 2               most common SCD
Only current value, typo corrections     SCD Type 1               overwrite in place
Track one prior value                    SCD Type 3               rare, limited
Fast time-range queries                  Partition by date         always do this
Fast joins by specific columns           Sort/cluster by join keys second priority
────────────────────────────────────────────────────────────────────────────

GRAIN DECISION:
  Atomic grain (lowest): use for operational/ad-hoc queries, drill-down analysis
  Aggregated grain:      use for dashboards with fixed dimensions, faster queries
  Rule: start atomic — you can always roll up, never drill down

MEASURE DECISION:
  Can you SUM it? → additive → store as-is
  Can you SUM it across some dims but not others? → semi-additive → use AVG or LAST
  Is it a ratio or rate? → non-additive → store components, compute at query time

SCD DECISION:
  Is history required for accurate reporting? → Type 2 (new row, close old)
  Is only current state needed?              → Type 1 (overwrite)
  Is exactly one prior value needed?         → Type 3 (add prev_column)
  Is very complex versioning needed?         → Type 6 = Type 2 + Type 3 combined
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for each pattern:

| Scenario | Pattern |
|---|---|
| Design analytics schema | Star Schema (fact + dimensions) |
| Track attribute history | SCD Type 2 (close old, insert new) |
| Fix typos, no history needed | SCD Type 1 (overwrite) |
| Speed up date-range queries | Partition by date |
| Speed up join columns | Cluster/sort by join key |

### The 5 DW design rules:

```
1. Declare grain FIRST — "one row represents one ____"
2. All facts must be true AT the grain level
3. Use surrogate keys (not natural keys) in all FK relationships
4. Store measures as additive values (numerator+denominator, not ratios)
5. SCD Type 2 for anything that needs historical accuracy
```

### SCD quick reference:

```
Type 1: UPDATE row in place — history lost, simple
Type 2: INSERT new row + close old — full history, most common
        CLOSE:  SET eff_end = today, is_current = 0
        INSERT: new row with eff_start = today, eff_end = 9999-12-31, is_current = 1
Type 3: ADD prev_X column — one prior value, limited history
```

### Gotchas to not forget:

```
❌  SUM(balance) across time periods — semi-additive, use LAST or snapshot
✅  Store balance as of a specific grain (end-of-day), then SUM across accounts only
❌  Store discount_pct as a measure — non-additive, don't SUM
✅  Store discount_amount (pct × revenue) — additive, safe to SUM
❌  Skip the surrogate key, use natural key as FK
✅  Surrogate keys decouple DW from source system changes + enable SCD Type 2
❌  Forget to close old SCD Type 2 row when inserting new version
✅  SCD2 load: CLOSE old row (eff_end, is_current=0) + INSERT new row atomically
❌  Partition by a high-cardinality column (user_id → millions of tiny files)
✅  Partition by time column (date, month) — bounded number of partitions
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
              DATA WAREHOUSE DESIGN
                        │
         ┌──────────────┼──────────────┐
         │              │              │
       SCHEMA        DIMENSIONS      PERFORMANCE
         │              │              │
      Star Schema    SCD Types      Partitioning
      (default)         │           by date
         │           Type 1: overwrite  │
      Snowflake      Type 2: new row  Clustering
      (hierarchies)  Type 3: add col  by join keys

      FACT TABLE DESIGN:
      ─────────────────────────────────────
      Grain: one row = one ___
      FKs: surrogate keys to all dims
      Measures: additive / semi / non-additive
      Degenerate dims: IDs with no dim table

MEASURE RULE:
  Additive → store as-is, SUM freely
  Semi-additive → use LAST / point-in-time
  Non-additive → store components, compute at query time
```

---
*End of Data Warehouse Design Master Guide — Sean Edition*